# Подготовка данных к моделированию

## Цель

На этом этапе подготовить данные для дальнейшего построения моделей кредитного скоринга.

Основные задачи:

- исключить технический идентификатор;
- разделить данные на обучающую и валидационную выборки;
- применить правила обработки обнаруженных аномальных значений;
- обработать пропущенные значения;
- стандартизировать числовые признаки;
- проверить корректность полученного preprocessing.

Все параметры preprocessing должны рассчитываться только на обучающей выборке,
чтобы избежать утечки информации из валидационной выборки.

In [1]:
# необходимые библеотеки
import sys
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

# Добавляем корень проекта в Python path
sys.path.append(str(Path.cwd().parent))

from src.preprocessing import preprocessor

In [2]:
# загружаем данные
train = pd.read_csv("../data/raw/cs-training.csv")

print(f"Размер датасета: {train.shape}")

Размер датасета: (150000, 12)


In [3]:
train.head()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [4]:
# выделяем целевую переменную
X = train.drop(columns=["SeriousDlqin2yrs", "Unnamed: 0"])
y = train["SeriousDlqin2yrs"]

In [5]:
# проверка признаков
print("Количество признаков:", X.shape[1])
print("Target:", y.name)

print("\nПризнаки:")
print(X.columns.tolist())

Количество признаков: 10
Target: SeriousDlqin2yrs

Признаки:
['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']


In [6]:
# разбиение на выборки
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
# размеры выборок
print("Train:", X_train.shape)
print("Validation:", X_valid.shape)

print("\nДоля положительного класса:")
print("Train:", y_train.mean())
print("Validation:", y_valid.mean())

Train: (120000, 10)
Validation: (30000, 10)

Доля положительного класса:
Train: 0.06684166666666666
Validation: 0.06683333333333333


In [8]:
# проверка пропусков до обработки
missing_before = pd.DataFrame({
    "train": X_train.isna().sum(),
    "validation": X_valid.isna().sum()
})

display(
    missing_before.query("train > 0 or validation > 0")
)

,train,validation
MonthlyIncome,23675,6056
NumberOfDependents,3128,796


In [9]:
# применяем preprocessing

X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

In [10]:
# проверка результата 

print("Train:", X_train_processed.shape)
print("Validation:", X_valid_processed.shape)

print("\nПропуски после preprocessing:")
print("Train:", np.isnan(X_train_processed).sum())
print("Validation:", np.isnan(X_valid_processed).sum())

Train: (120000, 10)
Validation: (30000, 10)

Пропуски после preprocessing:
Train: 0
Validation: 0


In [11]:
# проверяем стандартизацию 

processed_mean = X_train_processed.mean(axis=0)
processed_std = X_train_processed.std(axis=0)

print("Средние значения признаков:")
print(np.round(processed_mean, 3))

print("\nСтандартные отклонения:")
print(np.round(processed_std, 3))

Средние значения признаков:
[-0.  0. -0.  0.  0.  0. -0. -0. -0.  0.]

Стандартные отклонения:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## Итоги

В ходе подготовки данных:

- технический идентификатор `Unnamed: 0` исключён из признаков;
- данные разделены на обучающую и валидационную выборки в соотношении 80/20;
- при разделении сохранено исходное соотношение классов;
- значения `age = 0` преобразуются в пропуски;
- значения `96` и `98` в признаках истории просрочек преобразуются в пропуски;
- пропущенные значения заполняются медианами;
- числовые признаки стандартизируются;
- preprocessing обучается только на тренировочной выборке;
- к валидационной выборке применяется уже обученный preprocessing;
- после преобразования пропуски отсутствуют.

Подготовленный preprocessing реализован в `src/preprocessing.py` и может быть
повторно использован при обучении различных моделей.